# Spacecraft - TD3

**VecNormalize** scales the observations and rewards using running statistics, which helps with your large goal bonus.
It keeps those statistics in the vec env, so they are saved to a `.pkl` file next to the model and must be loaded again for evaluation.


In [9]:
pip install mujoco gymnasium stable-baselines3[extra] sympy psutil

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import TD3
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.noise import NormalActionNoise
import Spacecraft_V1

**Register the env**

In [11]:
gym.register(
    id="Spacecraft_V1",
    entry_point="Spacecraft_V1:SpacecraftEnv",
    max_episode_steps=2000,
)

c:\Users\texei\OneDrive\WindowsCode\EGR444_final\.venv\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment Spacecraft_V1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


**Declare the vectorized env (with VecNormalize) and the model**

In [12]:
# 6 copies of the env, all running in this process
vec_env = make_vec_env("Spacecraft_V1", n_envs=6, vec_env_cls=DummyVecEnv)

# normalize observations and rewards; clip_obs keeps the normalized observations in a sane range
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

# TD3 needs action noise, otherwise it only explores for the first 100 steps
n_actions = vec_env.action_space.shape[0]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))   # SB3 makes one noise stream per env

# gradient_steps=1 -> 1 update per 6 env steps (faster, less learning per step); -1 -> 1 update per env step
model = TD3("MlpPolicy", vec_env, action_noise=action_noise, verbose=1, device="cpu", gradient_steps=1)

Using cpu device


In [ ]:
model.learn(total_timesteps=2000000)

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1.03e+03  |
|    ep_rew_mean     | -1.86e+03 |
| time/              |           |
|    episodes        | 4         |
|    fps             | 489       |
|    time_elapsed    | 13        |
|    total_timesteps | 6612      |
| train/             |           |
|    actor_loss      | 0.131     |
|    critic_loss     | 0.00209   |
|    learning_rate   | 0.001     |
|    n_updates       | 1085      |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 906       |
|    ep_rew_mean     | -1.63e+03 |
| time/              |           |
|    episodes        | 8         |
|    fps             | 496       |
|    time_elapsed    | 18        |
|    total_timesteps | 9180      |
| train/             |           |
|    actor_loss      | 0.122     |
|    critic_loss     | 0.147     |
|    learning_rate   | 0.001     |
|    n_updates      

**Save the model and the normalization statistics**

In [ ]:
model.save("td3_norm_spacecraft")
vec_env.save("td3_norm_stats.pkl")   # the normalization statistics, needed to evaluate the model
vec_env.close()

**Evaluate model**

In [ ]:
model = TD3.load("td3_norm_spacecraft")
# the model trained on normalized observations, so load the saved statistics to normalize them the same way
stats = VecNormalize.load("td3_norm_stats.pkl", DummyVecEnv([lambda: gym.make("Spacecraft_V1")]))

eval_env = gym.make("Spacecraft_V1", render_mode="human")
obs, info = eval_env.reset()
for _ in range(2000):
    action, _states = model.predict(stats.normalize_obs(obs), deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    if terminated or truncated:
        obs, info = eval_env.reset()
eval_env.close()